# 06 - Online recalibration

A plant does not stand still: substrate quality changes with the harvest, the
biology adapts, a digester is emptied. A model calibrated in spring drifts away
from the plant by autumn.

`OnlineCalibrator` is built for that case. It differs from the initial
calibration in two ways:

- it starts from the **currently used** parameters instead of a default,
- it **refuses** to move them far, or at all, when the evidence is thin.

This notebook runs it next to a plant that keeps running and slowly changes. A
digital twin predicts the plant's gas. From time to time the twin is recalibrated
on the measurements of the last days and then used from that moment on.

## The plant and its twin

Two copies of the demo plant run side by side. `plant` stands in for the real one
and produces the measurements, `twin` is the model we recalibrate. Both first run
30 days on the standard feed, so they start from a settled digester. `advance()`
keeps the state a run ends in, so the next run continues from there.

**What changes.** The parameter is `k_dis_PF`, the disintegration rate of the
fast-degradable particulate fraction. In this plant it is the slowest step between
feed and gas, so a change in it shows up in the gas. (The hydrolysis rate
`k_hyd_ch` of the other notebooks acts within hours; a plant in steady operation
produces almost the same gas whatever its value.)

**The feed** changes every other day for half a day: a maize pulse, grass silage,
less manure. That is what a real plant gets, and how fast the gas follows a change
is what reveals the rate.

**The schedule** repeats one pattern three times:

| days | plant (`k_dis_PF`) | twin |
| --- | --- | --- |
| 0-4 | 0.40 | runs with a guess of 0.50 |
| **4** | | **calibration 1** on days 0-4 |
| 4-8 | unchanged | check: how good is the calibrated twin? |
| 8-10 | drifts to 0.30 | keeps its value |
| 10-12 | stays at 0.30 | keeps its value |
| **12** | | **calibration 2** on days 8-12 |
| 12-16 | unchanged | check |
| 16-18 | drifts to 0.22 | keeps its value |
| 18-20 | stays at 0.22 | keeps its value |
| **20** | | **calibration 3** on days 16-20 |
| 20-24 | unchanged | check |

Each calibration scores the **last 4 days** and is used **from that day on**. A
calibration only sees the past: it estimates the value that best explains those four
days. The change here takes two days, like a new silage batch phased in, and the plant
then stays there for two more. Were the plant still drifting over all four days, the
calibration could only find their average, and the twin would stay behind.

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from demo_plant import FEED, add_measurement_noise, advance, build_demo_plant, feed_frame
from matplotlib.patches import Patch

from pyadm1ode_calibration import MeasurementData, OnlineCalibrator

PARAMETER = "k_dis_PF"
GUESS = 0.50  # what the twin starts with
SCORED_DAYS, CHECK_DAYS = 4, 4
CALIBRATION_DAYS = [4, 12, 20]
# The plant: (day, value) corners, linear in between.
TRUE_CORNERS = [(0, 0.40), (8, 0.40), (10, 0.30), (16, 0.30), (18, 0.22), (24, 0.22)]
DAYS = CALIBRATION_DAYS[-1] + CHECK_DAYS


def true_value(day):
    """The plant's k_dis_PF on a given (fractional) day."""
    corner_days, corner_values = zip(*TRUE_CORNERS)
    return float(np.interp(day, corner_days, corner_values))


# Every other day, half a day on a different mix [maize, manure, grass] in m3/d.
MIXES = [[30, 10, 0], [15, 10, 15], [22, 10, 8], [30, 5, 0]]
changes = []
for day in range(1, DAYS, 2):
    changes += [(day, MIXES[(day // 2) % len(MIXES)]), (day + 0.5, FEED)]

plant = build_demo_plant(days=30 + DAYS)  # stands in for the real plant
twin = build_demo_plant(days=30 + DAYS)  # the model we calibrate

# A common history: both settle on the standard feed.
start = pd.Timestamp("2024-01-01")
settle = feed_frame(30, start)
advance(plant, settle, {PARAMETER: true_value(0)})
advance(twin, settle, {PARAMETER: true_value(0)})
start += pd.Timedelta(days=30)

# The plant runs through the whole record; its rate moves every 6 hours.
feed = feed_frame(DAYS, start, changes)
STEP_HOURS = 6
chunks = []
for first in range(0, len(feed), STEP_HOURS):
    chunk = feed.iloc[first : first + STEP_HOURS].reset_index(drop=True)
    chunks.append(advance(plant, chunk, {PARAMETER: true_value(first / 24)}))
outputs = {name: np.concatenate([np.asarray(c[name], dtype=float) for c in chunks]) for name in chunks[0]}
record = add_measurement_noise(feed, outputs, noise=0.02, rng=np.random.default_rng(10)).data
measured = record["Q_gas"].to_numpy()


def days(first, last):
    """The measurements from the start of day `first` to the start of day `last`."""
    return MeasurementData(record.iloc[first * 24 : last * 24].reset_index())

## Running the twin

Every day the twin predicts the day with the value it currently has. On a
calibration day it is recalibrated first:

- **Data**: the last 4 days. Every candidate value is simulated from the twin's state
  at the start of those days. The twin keeps a copy of its state for every day, so
  that state is at hand. It can be trusted: in the check window before, the twin
  matched the plant. Calibration 1 starts from the settled state both share.
- **Afterwards** the twin runs those four days again with the new value, so its state
  agrees with it, and predicts from today on with the new value.

For comparison, the twin that kept its old value also predicts the check window.

In [ ]:
current = {PARAMETER: GUESS}
used = [(0, GUESS)]  # (from day, value)
saved_state = {}  # day -> copy of the twin at the start of that day
prediction = np.zeros(DAYS * 24)  # what the twin predicted, day by day
kept_old = {}  # calibration day -> prediction of the check window with the old value

for day in range(DAYS):
    if day in CALIBRATION_DAYS:
        first = day - SCORED_DAYS
        result = OnlineCalibrator(copy.deepcopy(saved_state[first]), verbose=False, time_varying_feed=True).calibrate(
            days(first, day),
            parameters=[PARAMETER],
            current_parameters=current,
            objectives=["Q_gas"],
            max_parameter_change=0.30,
            max_iterations=15,
            time_window=SCORED_DAYS,
            use_constraints=False,
        )
        old = current
        kept_old[day] = advance(copy.deepcopy(twin), days(day, day + CHECK_DAYS), old)["Q_gas"]
        current = dict(result.parameters)
        used.append((day, current[PARAMETER]))
        print(
            f"day {day:2d}: {PARAMETER} {old[PARAMETER]:.3f} -> {current[PARAMETER]:.3f}"
            f"   (plant over the scored days: {true_value(day - SCORED_DAYS):.2f} -> {true_value(day):.2f})"
        )
        # Re-run the recent past with the new value, so the twin's state agrees with it.
        twin = copy.deepcopy(saved_state[first])
        for past in range(first, day):
            saved_state[past] = copy.deepcopy(twin)
            advance(twin, days(past, past + 1), current)

    saved_state[day] = copy.deepcopy(twin)
    hours = slice(day * 24, (day + 1) * 24)
    prediction[hours] = np.asarray(advance(twin, days(day, day + 1), current)["Q_gas"], dtype=float)

## What happens

The upper panel shows the measured gas and the twin's prediction; in the check
windows (green) also the prediction a twin would have made that kept its old value.
The lower panel shows the plant's true `k_dis_PF` and the value the twin uses. Blue
bands: the four days a calibration uses.

In [ ]:
t = np.arange(DAYS * 24) / 24  # days

fig, (ax, ax_k) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
for n, day in enumerate(CALIBRATION_DAYS, start=1):
    for axis in (ax, ax_k):
        axis.axvspan(day - SCORED_DAYS, day, color="tab:blue", alpha=0.10, lw=0)
        axis.axvspan(day, day + CHECK_DAYS, color="tab:green", alpha=0.10, lw=0)
        axis.axvline(day, color="tab:blue", lw=1)
    ax.text(day, 1.01, f"calibration {n}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=9)
    check = slice(day * 24, (day + CHECK_DAYS) * 24)
    ax.plot(t[check], kept_old[day], color="tab:red", ls="--", lw=1.4,
            label="twin with its old value" if n == 1 else None)

ax.plot(t, measured, color="0.55", lw=0.6, label="measured")
ax.plot(t, prediction, color="tab:blue", lw=1.8, label="twin")
ax.set_ylabel("Q_gas [m3/d]")
handles, _ = ax.get_legend_handles_labels()
handles += [
    Patch(color="tab:blue", alpha=0.10, label="data of a calibration"),
    Patch(color="tab:green", alpha=0.10, label="check window"),
]
ax.legend(handles=handles, fontsize=8, loc="lower right", ncol=3)

ax_k.plot(t, [true_value(d) for d in t], color="tab:green", ls="--", lw=1.6, label="plant (true)")
starts = [d for d, _ in used] + [DAYS]
values = [v for _, v in used]
ax_k.step(starts, values + values[-1:], where="post", color="tab:blue", lw=1.8, label="twin")
ax_k.set_ylabel(PARAMETER + " [1/d]")
ax_k.set_xlabel("day")
ax_k.set_xlim(0, DAYS)
ax_k.legend(fontsize=8, loc="upper right")
fig.tight_layout()


def rmse(y, hours):
    return float(np.sqrt(np.mean((np.asarray(y) - measured[hours]) ** 2)))


print("RMSE of Q_gas in the check windows [m3/d]")
for n, day in enumerate(CALIBRATION_DAYS, start=1):
    check = slice(day * 24, (day + CHECK_DAYS) * 24)
    print(f"  after calibration {n}: old value {rmse(kept_old[day], check):6.1f}   recalibrated {rmse(prediction[check], check):6.1f}")

Read it from left to right:

- **Calibration 1** corrects the guess of 0.50 to 0.39. In the check window the twin
  follows the measurement about as closely as the noise allows.
- **During a drift** the twin keeps its value, and the measured gas falls away from
  its prediction.
- **Calibrations 2 and 3** see the drift and the days after it. They land slightly
  above the plant's new value (0.32 for 0.30, 0.25 for 0.22): the two drift days, in
  which the plant still had a higher value, pull the estimate up a little. In the check
  window the error is 40 to 50 % below that of the old value.

The noise is 2 % per hourly value, about 60 m3/d. An RMSE near that is as good as
any model can match these measurements.

## Where this stops working

The calibrations here work because every one of them starts from a twin state that
matches the plant. Its limits:

- **A state that is already wrong.** If the plant changed before the four days, or
  slow material (`k_dis_PS` is 0.04 per day) has drifted for weeks, the twin's state
  no longer matches at the start of the data. The calibration then fits the parameter
  to that wrong state and typically stays near the old value.
- **Drift during the scored days.** The estimate is a compromise over a moving target
  and lags behind it. A plant drifting through all four days gives roughly their
  average.
- **One observed quantity.** Everything here is inferred from the gas flow. Two
  parameters with a similar effect on the gas cannot be told apart this way.

In operation, `OnlineCalibrator.should_recalibrate` and its `trigger` settings decide
when a recalibration is due, instead of the fixed days used here. The remedy for the
limits above is to estimate the state along with the parameters, for instance with a
Kalman filter as in [PyADM1ODE_estimate](https://github.com/dgaida/PyADM1ODE_estimate).